# S01 - CAPE excess yield (reference trial)

**Status: REFERENCE.** One fixed rule, no parameter search. Its job is to show the
full Stage 1 pipeline end to end (point-in-time alignment, look-ahead test, three
frozen periods, bootstrap vs SMA200, deflated Sharpe, registry row) on a signal
everyone knows. Pass or fail is reported as is.

## Rule

* `ey_t = 1 / CAPE_t - GS10_t / 100` - Shiller cyclically adjusted earnings yield
  minus the 10-year Treasury yield (both from the monthly Shiller file).
* Long equities when `ey` is above its own **expanding median** (first 10 years
  used as warm-up, median lagged one month); otherwise 3-month T-bills.
* Monthly check, next-close execution, 5 bp per switch - identical to the SMA200
  benchmark so the comparison is like for like.

## Point-in-time convention

Shiller publishes month *M* roughly one month late, and the Shiller `P` for month
*M* is itself a monthly average. We therefore make the month-*M* observation usable
only from the first trading day of **M+2** (`bt_pit.RELEASE_LAGS["SHILLER_CAPE"]`,
one extra month on top of the usual M+1). The raw S&P series ends 2024-09 in the
file used here, so the signal is frozen (forward-filled) from 2024-11 onward; the
holdout period therefore includes ~2 years of a stale signal, which is noted below.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
sys.path.insert(0, str(_root / "notebooks" / "backtest"))
import bt_common as bc  # noqa: E402
import bt_long  # noqa: E402
import bt_periods as bp  # noqa: E402
import bt_pit  # noqa: E402
import bt_registry as br  # noqa: E402
import bt_stats as bs  # noqa: E402
import plotly.graph_objects as go  # noqa: E402

TRIAL_ID = "S01"
STRATEGY = "cape_excess_yield"
NOTEBOOK = "valuation/cape_excess_yield.ipynb"
PARAMS = "ey=1/CAPE-GS10; rule=ey>expanding_median(min 120m, lag 1m); pit M+2; monthly; 5bp"
WARMUP_MONTHS = 120
LAG = bt_pit.RELEASE_LAGS["SHILLER_CAPE"]

sample = bt_long.long_sample()
px, cash, shiller = sample["px"], sample["cash"], sample["shiller"]
print("prices", px.index[0].date(), "->", px.index[-1].date(), "| shiller ends", shiller.index[-1])

prices 1934-02-01 -> 2026-10-19 | shiller ends 2024-09


## 1. Signal construction (monthly, then point-in-time daily)

In [2]:
m = shiller[["CAPE", "GS10"]].dropna().copy()
m.index = m.index.to_timestamp(how="start")
ey = (1.0 / m["CAPE"] - m["GS10"] / 100.0).rename("ey")
median = ey.expanding(min_periods=WARMUP_MONTHS).median().shift(1)
sig_m = (ey > median).astype(float)
sig_m[median.isna()] = np.nan

# month M observation usable from first trading day of M+2
signal = bt_pit.monthly_to_daily_pit(sig_m, px.index, **LAG).rename("signal")
ey_daily = bt_pit.monthly_to_daily_pit(ey, px.index, **LAG).rename("ey")
median_daily = bt_pit.monthly_to_daily_pit(median, px.index, **LAG).rename("ey_median")

first = signal.first_valid_index()
switches = int((signal.diff().abs() > 0).sum())
print(f"first signal {first.date()} | share in equities {signal.mean():.1%} | monthly switches {switches}")
print("last CAPE month in file:", m.index[-1].date(), "-> signal frozen after", (m.index[-1] + pd.offsets.MonthBegin(2)).date())
display(pd.concat([ey, median.rename("median"), sig_m.rename("signal")], axis=1).tail(6))

first signal 1934-02-01 | share in equities 26.4% | monthly switches 23
last CAPE month in file: 2024-09-01 -> signal frozen after 2024-11-01


,ey,median,signal
2024-04-01,-0.015226,0.020532,0.0
2024-05-01,-0.015191,0.020523,0.0
2024-06-01,-0.014373,0.020513,0.0
2024-07-01,-0.014286,0.020495,0.0
2024-08-01,-0.010133,0.020477,0.0
2024-09-01,-0.010014,0.020457,0.0


In [3]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ey.index, y=ey * 100, name="excess yield (%)"))
fig.add_trace(go.Scatter(x=median.index, y=median * 100, name="expanding median (%)", line=dict(dash="dash")))
fig.add_trace(go.Scatter(x=sig_m.index, y=sig_m * 10 - 15, name="signal (1 = equities, offset)", line=dict(width=1)))
fig.update_layout(title="CAPE excess yield vs expanding median", height=420, legend=dict(orientation="h"))
fig.show()

## 2. Look-ahead test

Shift the daily signal by -21 .. +21 trading days. If *peeking* (negative shifts)
does **not** improve the Sharpe, the signal as aligned may already contain future
information. We expect a clear gain from peeking for a genuinely lagged signal.

In [4]:
la = bt_pit.lookahead_test(px, signal, cash)
display(la.to_frame().T.round(3))
print(f"peek gain (Sharpe at -21d minus at 0): {la.peek_gain:+.3f} | suspicious: {la.suspicious}")

,-21,-5,-1,0,1,5,21
Sharpe,0.44,0.485,0.485,0.46,0.456,0.463,0.481


peek gain (Sharpe at -21d minus at 0): -0.019 | suspicious: True


## 3. Backtest per frozen period vs benchmarks

Periods and benchmarks come from `bt_periods` (dev 1934-02..1989-12, val 1990..2009,
holdout 2010.., full). The holdout is confirmatory only.

In [5]:
runs = bp.run_periods(px, signal, cash, extra={"ey": ey_daily, "ey_median": median_daily})
bench = bp.benchmark_returns(px, cash)
cmp = bp.compare_to_benchmarks(runs, bench)
display(cmp.round(3))
stats = pd.DataFrame({p: bc.perf_stats(df["r_strat"], df["r_cash"]) for p, df in runs.items()}).T
display(stats)

,strategy,bh,d_bh,sma200,d_sma200,mix60,d_mix60
dev,0.573,0.511,0.062,0.644,-0.071,0.518,0.055
val,0.139,0.319,-0.180,0.572,-0.433,0.309,-0.170
holdout,0.649,0.781,-0.132,0.658,-0.009,0.779,-0.130
full,0.503,0.513,-0.010,0.630,-0.127,0.515,-0.012


,CAGR,Vol,Sharpe,Sortino,Max DD,Calmar,Worst day,MaxDD peak,MaxDD recovered,Final x
dev,0.097265,0.10562,0.573147,0.820833,-0.337574,0.28813,-0.099477,1938-10-10,1943-02-11,163.171799
val,0.044287,0.059351,0.139304,0.201613,-0.271852,0.162907,-0.052808,2009-01-06,2009-05-08,2.37843
holdout,0.054662,0.062724,0.648725,0.941574,-0.09582,0.570468,-0.058829,2012-04-02,2012-08-16,2.443644
full,0.077622,0.090353,0.503181,0.722241,-0.337574,0.229942,-0.099477,1938-10-10,1943-02-11,948.360172


In [6]:
full = runs["full"]
eq = (1 + full["r_strat"]).cumprod()
eq_bh = (1 + full["r_bh"]).cumprod()
sma = (1 + bench["sma200"].reindex(full.index).fillna(0)).cumprod()
fig = go.Figure()
fig.add_trace(go.Scatter(x=eq.index, y=eq, name="S01 CAPE excess yield"))
fig.add_trace(go.Scatter(x=eq_bh.index, y=eq_bh, name="buy & hold"))
fig.add_trace(go.Scatter(x=sma.index, y=sma, name="SMA200 benchmark"))
for name, (a, b) in bp.PERIODS.items():
    fig.add_shape(type="line", x0=a, x1=a, y0=0, y1=1, yref="paper", line=dict(color="grey", dash="dot"))
    fig.add_annotation(x=a, y=1, yref="paper", text=name, showarrow=False, yanchor="bottom")
fig.update_layout(title="Growth of 1 (log scale)", yaxis_type="log", height=480, legend=dict(orientation="h"))
fig.show()

## 4. Primary metric (frozen) and robustness

* Pass rule: Sharpe > SMA200 **and** > B&H in dev and val, one-sided block-bootstrap
  p <= 0.10 vs SMA200 in both, deflated Sharpe >= 0.95 using the registry trial
  count (this trial included).
* Cost sensitivity and a placebo test (signal dates shuffled in blocks) follow.

In [7]:
n_trials = br.trial_count() + 1  # registered so far plus this one
res = bs.primary_metric(runs, bench, n_trials=n_trials, rng=np.random.default_rng(bs.SEED))
display(res.to_frame().round(3))
print(f"n_trials={res.n_trials} | DSR={res.dsr:.3f} | passes={res.passes} | holdout confirms={res.holdout_confirms}")
display(pd.Series(res.checks, name="check"))

,Sharpe,d_sma200,d_bh,p_vs_sma200
dev,0.573,-0.071,0.062,0.782
val,0.139,-0.433,-0.180,0.965
holdout,0.649,-0.009,-0.132,NaN
full,0.503,-0.127,-0.010,NaN


n_trials=5 | DSR=1.000 | passes=False | holdout confirms=False


dev_beats_sma200        False
dev_beats_bh             True
dev_p_le_0.1            False
val_beats_sma200        False
val_beats_bh            False
val_p_le_0.1            False
dsr_ge_0.95              True
holdout_d_sma200_pos    False
Name: check, dtype: bool

In [8]:
cs = bs.cost_sensitivity(px, signal, cash)
display(cs.round(3))
bench_full = bc.perf_stats(bench["sma200"], bench["tbill"])["Sharpe"]
be = bs.breakeven_cost_bps(px, signal, cash, bench_sharpe=bench_full)
print(f"break-even cost to match SMA200 full-sample Sharpe ({bench_full:.3f}): {be:.1f} bp")

,Sharpe,CAGR,Max DD,turnover_pa
cost_bps,,,,
0,0.461,0.074,-0.338,0.26
5,0.460,0.074,-0.338,0.26
10,0.458,0.074,-0.338,0.26
25,0.454,0.074,-0.338,0.26
50,0.447,0.073,-0.338,0.26


break-even cost to match SMA200 full-sample Sharpe (0.630): 0.0 bp


In [9]:
pl = bs.placebo_test(px, signal, cash, n_placebo=200, rng=np.random.default_rng(bs.SEED))
display(pd.DataFrame([pl]).T.rename(columns={0: "placebo"}))

,placebo
stat,Sharpe
real,0.459806
placebo_mean,0.251567
placebo_q95,0.434256
quantile,0.98
p_one_sided,0.02
n_placebo,200
time_in_market,0.263991


## 5. Register the trial

The planned placeholder row `S01` is replaced by one row per period
(`S01/dev`, `S01/val`, `S01/holdout`, `S01/full`). The status is whatever the
frozen metric says; a REFERENCE trial that fails is still recorded.

In [10]:
import seed_registry

reg = br.load_registry()
reg = reg[reg["trial_id"] != TRIAL_ID]
reg.to_csv(br.REGISTRY_CSV, index=False, lineterminator="\n")

if res.passes:
    note = "REFERENCE trial; passes frozen metric"
else:
    note = "REFERENCE trial; fails frozen metric (does not beat SMA200 in dev/val); Shiller file ends 2024-09 so holdout tail is a frozen signal"
rows = br.rows_from_primary(TRIAL_ID, STRATEGY, NOTEBOOK, PARAMS, res, note=note)
br.append_trials(rows)
br.render_registry(title=seed_registry.HEADER)
print("registry trial count:", br.trial_count())
display(br.load_registry().query("trial_id.str.startswith(@TRIAL_ID)", engine="python"))

registry trial count: 5


,trial_id,date,strategy,notebook,params,period,sharpe,d_sma200,d_bh,p_vs_sma200,dsr,n_trials,passes,status,note
25,S01/dev,2026-10-20,cape_excess_yield,valuation/cape_excess_yield.ipynb,ey=1/CAPE-GS10; rule=ey>expanding_median(min 1...,dev,0.5731,-0.0706,0.0621,0.7822,,5,False,fail,REFERENCE trial; fails frozen metric (does not...
26,S01/val,2026-10-20,cape_excess_yield,valuation/cape_excess_yield.ipynb,ey=1/CAPE-GS10; rule=ey>expanding_median(min 1...,val,0.1393,-0.4329,-0.1797,0.9655,,5,False,fail,REFERENCE trial; fails frozen metric (does not...
27,S01/holdout,2026-10-20,cape_excess_yield,valuation/cape_excess_yield.ipynb,ey=1/CAPE-GS10; rule=ey>expanding_median(min 1...,holdout,0.6487,-0.0088,-0.1320,,,5,False,fail,REFERENCE trial; fails frozen metric (does not...
28,S01/full,2026-10-20,cape_excess_yield,valuation/cape_excess_yield.ipynb,ey=1/CAPE-GS10; rule=ey>expanding_median(min 1...,full,0.5032,-0.1265,-0.0101,,0.9998,5,False,fail,REFERENCE trial; fails frozen metric (does not...


## 6. Interpretation

* The excess-yield rule is a slow valuation timer: it sits in equities only about a
  quarter of the time and switches a few dozen times in ninety years.
* It beats buy & hold in the development period but not the SMA200 benchmark, and it
  trails both in validation, so it **fails the frozen primary metric**. That is the
  expected outcome for a reference trial: it anchors the registry and sets the bar for
  Stage 2 ideas, which must beat SMA200 in dev *and* val.
* The look-ahead test is flagged `suspicious` (peek gain -0.02): shifting the daily signal
  by +/-21 days barely moves the Sharpe (0.44-0.49). With only ~23 switches in ninety
  years the test has almost no power for this signal, so the flag is uninformative rather
  than evidence of leakage; the M+2 point-in-time lag is applied explicitly in section 1.
* Holdout figures are reported for completeness only and do not enter the pass rule.
  Shiller data ends 2024-09, so from 2024-11 the signal is frozen at its last value.